In [2]:
# Nama: Fajira Zahara
# NIM: 24343033
# Class Code: 202523430039

import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.utils import to_categorical

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import seaborn as sns

# 1. KONFIGURASI GLOBAL
IMG_SIZE    = 32
BATCH_SIZE  = 64
EPOCHS_BASE = 20
EPOCHS_TL   = 15
EPOCHS_FT   = 10
NUM_CLASSES = 10
SEED        = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

# 2. MEMUAT DAN PRA-PEMROSESAN DATASET (CIFAR-10)
def load_dataset():
    print("=" * 60)
    print("LANGKAH 1: MEMUAT DATASET CIFAR-10")
    print("=" * 60)

    (X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

    y_train = y_train.flatten()
    y_test  = y_test.flatten()

    print(f"Jumlah data training : {len(X_train)}")
    print(f"Jumlah data testing  : {len(X_test)}")
    print(f"Ukuran citra         : {X_train.shape[1:]} piksel")
    print(f"Jumlah kelas         : {NUM_CLASSES}")

    # Normalisasi
    X_train_norm = X_train.astype('float32') / 255.0
    X_test_norm  = X_test.astype('float32')  / 255.0

    # One-hot encoding
    y_train_cat = to_categorical(y_train, NUM_CLASSES)
    y_test_cat  = to_categorical(y_test, NUM_CLASSES)

    return (X_train_norm, y_train, y_train_cat,
            X_test_norm,  y_test,  y_test_cat)


# 3. VISUALISASI SAMPEL DATASET
def visualize_dataset(X_train, y_train):
    print("\nMenunjukkan sampel dataset...")

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle("Sampel Dataset CIFAR-10", fontsize=14, fontweight='bold')

    for i, ax in enumerate(axes.flat):
        idx = np.where(y_train == i)[0][0]
        ax.imshow(X_train[idx])
        ax.set_title(CLASS_NAMES[i], fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


# 4. DATA AUGMENTATION
def build_augmentation():
    print("\n" + "=" * 60)
    print("LANGKAH 2: DATA AUGMENTATION")
    print("=" * 60)

    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        zoom_range=0.2,
        shear_range=0.2,
        fill_mode='nearest'
    )

    return datagen


def visualize_augmentation(datagen, X_train):
    sample = X_train[0:1]

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle("Visualisasi Hasil Data Augmentasi",
                 fontsize=14,
                 fontweight='bold')

    axes[0, 0].imshow(X_train[0])
    axes[0, 0].set_title("Original", fontsize=9)
    axes[0, 0].axis('off')

    gen = datagen.flow(sample, batch_size=1)

    for i, ax in enumerate(axes.flat):
        if i == 0:
            continue

        batch = next(gen)
        ax.imshow(np.clip(batch[0], 0, 1))
        ax.set_title(f"Augmented {i}", fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


# 5. CNN FROM SCRATCH
def build_cnn_scratch(optimizer_name='adam', lr=0.001):

    if optimizer_name == 'adam':
        opt = optimizers.Adam(learning_rate=lr)
    else:
        opt = optimizers.SGD(learning_rate=lr, momentum=0.9)

    model = models.Sequential([

        layers.Conv2D(
            32,
            (3,3),
            activation='relu',
            padding='same',
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        ),

        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Flatten(),

        layers.Dense(256, activation='relu'),

        layers.Dropout(0.5),

        layers.Dense(NUM_CLASSES, activation='softmax')

    ], name=f"CNN_Scratch_{optimizer_name}")

    model.compile(
        optimizer=opt,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


def train_cnn_scratch(X_train, y_train_cat, X_test, y_test_cat, datagen):

    print("\n" + "=" * 60)
    print("LANGKAH 3: TRAINING CNN FROM SCRATCH")
    print("=" * 60)

    results = {}

    for opt_name, lr in [('adam', 0.001), ('sgd', 0.01)]:

        print(f"\nTraining dengan optimizer: {opt_name.upper()} (lr={lr})")

        model = build_cnn_scratch(
            optimizer_name=opt_name,
            lr=lr
        )

        cb = [
            callbacks.EarlyStopping(
                patience=5,
                restore_best_weights=True
            ),

            callbacks.ReduceLROnPlateau(
                patience=3,
                factor=0.5,
                verbose=0
            )
        ]

        t0 = time.time()

        history = model.fit(
            datagen.flow(
                X_train,
                y_train_cat,
                batch_size=BATCH_SIZE
            ),
            epochs=EPOCHS_BASE,
            validation_data=(X_test, y_test_cat),
            callbacks=cb,
            verbose=1
        )

        elapsed = time.time() - t0

        _, acc = model.evaluate(
            X_test,
            y_test_cat,
            verbose=0
        )

        print(f"Accuracy ({opt_name}): {acc:.4f}")
        print(f"Waktu training: {elapsed:.1f} detik")

        results[opt_name] = {
            'model': model,
            'history': history,
            'acc': acc,
            'time': elapsed
        }

    return results


# 6. TRANSFER LEARNING
def build_transfer_model(base_name, fine_tune=False):

    if base_name == 'vgg16':

        base = VGG16(
            weights='imagenet',
            include_top=False,
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )

    elif base_name == 'resnet50':

        base = ResNet50(
            weights='imagenet',
            include_top=False,
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )

    else:

        base = MobileNetV2(
            weights='imagenet',
            include_top=False,
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )

    base.trainable = fine_tune

    if fine_tune:
        for layer in base.layers[:-20]:
            layer.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = models.Model(
        inputs=base.input,
        outputs=out,
        name=f"TL_{base_name}_{'finetune' if fine_tune else 'feature'}"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=1e-4 if not fine_tune else 1e-5
        ),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


def train_transfer_learning(X_train,
                            y_train_cat,
                            X_test,
                            y_test_cat,
                            datagen):

    print("\n" + "=" * 60)
    print("LANGKAH 4: TRAINING TRANSFER LEARNING")
    print("=" * 60)

    tl_results = {}

    for base_name in ['mobilenetv2', 'vgg16', 'resnet50']:

        for fine_tune in [False, True]:

            label = f"{base_name}_{'finetune' if fine_tune else 'feature'}"

            epochs = EPOCHS_FT if fine_tune else EPOCHS_TL

            print(f"\nTraining: {label}")

            model = build_transfer_model(base_name, fine_tune)

            cb = [
                callbacks.EarlyStopping(
                    patience=4,
                    restore_best_weights=True
                )
            ]

            t0 = time.time()

            history = model.fit(
                datagen.flow(
                    X_train,
                    y_train_cat,
                    batch_size=BATCH_SIZE
                ),
                epochs=epochs,
                validation_data=(X_test, y_test_cat),
                callbacks=cb,
                verbose=1
            )

            elapsed = time.time() - t0

            _, acc = model.evaluate(X_test, y_test_cat, verbose=0)

            print(f"Accuracy: {acc:.4f}")
            print(f"Waktu training: {elapsed:.1f} detik")

            tl_results[label] = {
                'model': model,
                'history': history,
                'acc': acc,
                'time': elapsed
            }

    return tl_results


# 7. EVALUASI MODEL
def evaluate_model(model, X_test, y_test, y_test_cat, label):

    print(f"\nEvaluasi model: {label}")

    y_pred_prob = model.predict(X_test, verbose=0)

    y_pred = np.argmax(y_pred_prob, axis=1)

    print(classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES
    ))

    # CONFUSION MATRIX
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(9, 7))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax
    )

    ax.set_title(
        f"Confusion Matrix - {label}",
        fontsize=13,
        fontweight='bold'
    )

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    plt.tight_layout()
    plt.show()

    # ROC CURVE
    fig, ax = plt.subplots(figsize=(9, 7))

    for i in range(NUM_CLASSES):

        fpr, tpr, _ = roc_curve(
            y_test_cat[:, i],
            y_pred_prob[:, i]
        )

        roc_auc = auc(fpr, tpr)

        ax.plot(
            fpr,
            tpr,
            lw=1.5,
            label=f"{CLASS_NAMES[i]} (AUC={roc_auc:.2f})"
        )

    ax.plot([0,1],[0,1],'k--', lw=1)

    ax.set_title(
        f"ROC Curve - {label}",
        fontsize=13,
        fontweight='bold'
    )

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")

    ax.legend(loc='lower right', fontsize=8)

    plt.tight_layout()
    plt.show()

    return y_pred, y_pred_prob


# 8. LEARNING CURVE
def plot_learning_curve(history_dict, title):

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    fig.suptitle(title, fontsize=13, fontweight='bold')

    for label, h in history_dict.items():

        hist = h['history'].history

        axes[0].plot(
            hist['accuracy'],
            label=f"{label} train"
        )

        axes[0].plot(
            hist['val_accuracy'],
            label=f"{label} val",
            linestyle='--'
        )

        axes[1].plot(
            hist['loss'],
            label=f"{label} train"
        )

        axes[1].plot(
            hist['val_loss'],
            label=f"{label} val",
            linestyle='--'
        )

    for ax, title_ax in zip(axes, ['Accuracy', 'Loss']):

        ax.set_title(title_ax)

        ax.set_xlabel("Epoch")

        ax.legend(fontsize=8)

        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# MAIN
def main():

    print("=" * 60)
    print("PRAKTIKUM 14: CNN - DARI AWAL HINGGA TRANSFER LEARNING")
    print("Nama  : Fajira Zahara | NIM: 24343033")
    print("=" * 60)

    # 1. Muat dataset
    (X_train, y_train, y_train_cat,
     X_test,  y_test,  y_test_cat) = load_dataset()

    # 2. Visualisasi dataset
    visualize_dataset(X_train, y_train)

    # 3. Augmentasi
    datagen = build_augmentation()

    visualize_augmentation(datagen, X_train)

    # 4. CNN from scratch
    scratch_results = train_cnn_scratch(
        X_train,
        y_train_cat,
        X_test,
        y_test_cat,
        datagen
    )

    plot_learning_curve(
        scratch_results,
        "Learning Curve - CNN From Scratch"
    )

    # 5. Transfer learning
    tl_results = train_transfer_learning(
        X_train,
        y_train_cat,
        X_test,
        y_test_cat,
        datagen
    )

    plot_learning_curve(
        tl_results,
        "Learning Curve - Transfer Learning"
    )

    # 6. Evaluasi model terbaik CNN scratch (adam)
    best_scratch = scratch_results['adam']['model']

    print("\n" + "=" * 60)
    print("LANGKAH 5: EVALUASI MODEL")
    print("=" * 60)

    evaluate_model(
        best_scratch,
        X_test,
        y_test,
        y_test_cat,
        "CNN_Scratch_Adam"
    )

    # 7. Evaluasi model TL terbaik
    best_tl_key = max(
        tl_results,
        key=lambda k: tl_results[k]['acc']
    )

    best_tl = tl_results[best_tl_key]['model']

    evaluate_model(
        best_tl,
        X_test,
        y_test,
        y_test_cat,
        best_tl_key
    )

    # 8. Ringkasan hasil
    all_results = {
        **scratch_results,
        **tl_results
    }

    print("\n" + "=" * 60)
    print("RINGKASAN HASIL EKSPERIMEN")
    print("=" * 60)

    print(f"{'Model':<35} {'Accuracy':>10} {'Waktu (s)':>12}")

    print("-" * 60)

    for k, v in all_results.items():

        print(
            f"{k:<35} "
            f"{v['acc']*100:>8.2f}% "
            f"{v['time']:>10.1f}s"
        )

    print("=" * 60)

    print("\nSelesai!")


if __name__ == "__main__":
    main()

PRAKTIKUM 14: CNN - DARI AWAL HINGGA TRANSFER LEARNING
Nama  : Fajira Zahara | NIM: 24343033
LANGKAH 1: MEMUAT DATASET CIFAR-10
Jumlah data training : 50000
Jumlah data testing  : 10000
Ukuran citra         : (32, 32, 3) piksel
Jumlah kelas         : 10

Menunjukkan sampel dataset...

LANGKAH 2: DATA AUGMENTATION

LANGKAH 3: TRAINING CNN FROM SCRATCH

Training dengan optimizer: ADAM (lr=0.001)
Epoch 1/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.3315 - loss: 1.8366 - val_accuracy: 0.3749 - val_loss: 1.8786 - learning_rate: 0.0010
Epoch 2/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 39ms/step - accuracy: 0.4339 - loss: 1.5641 - val_accuracy: 0.5076 - val_loss: 1.3956 - learning_rate: 0.0010
Epoch 3/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 39ms/step - accuracy: 0.4946 - loss: 1.4217 - val_accuracy: 0.5401 - val_loss: 1.3487 - learning_rate: 0.0010
Epoch 4/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 38ms/step - accuracy: 0.5249 - loss: 1.3249 - val_accuracy: 0.6251 - val_loss: 1.0530 - learning_